In [19]:
import sys
import os

parent_dir = os.path.abspath('..')
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from pathlib import Path
import importlib

import numpy as np
import pandas as pd
from astropy.io import ascii
import matplotlib.pyplot as plt

from dask.distributed import Client
import dask.array
from dask.dataframe.utils import make_meta

from hats import read_hats
from hats.inspection import plot_pixels
from hats_import.catalog.file_readers import CsvReader
from hats_import.margin_cache.margin_cache_arguments import MarginCacheArguments
from hats_import.pipeline import ImportArguments, pipeline_with_client

import lsdb

from catalog_filtering import bandFilterLenient, contains_PM
import hpms_pipeline_test_2 as hpms

print("Imported libraries.")

Imported libraries.


In [2]:
# Change to the directories where the data will be stored
PERSONAL_CATALOG_DIR = Path("../../../../catalogs")
CATALOG_DIR = Path("../../../../shared/hats/catalogs/ps1")

GAIA_NAME = "gaia_dr3_pm_greater_100"
GAIA_DIR = PERSONAL_CATALOG_DIR / GAIA_NAME

ps1_name = "ps1_otmo"
DES_DIR = CATALOG_DIR / DES_NAME 

print("Defined directories.")

des_catalog = lsdb.read_hats(DES_DIR)
gaia_catalog = lsdb.read_hats(GAIA_DIR)
xmatch_catalog = lsdb.read_hats(DES_X_GAIA_DIR, columns='all')
print("Defined catalogs.")

Defined directories.
Defined catalogs.


In [3]:
bandList = ['G','R','I','Z','Y']
class_star = None
spread_model = 0.05
magnitude_error = 0.05
check_flags = True
mag = 19
query_string = bandFilterLenient(bandList,classStar=class_star,spreadModel=spread_model,magError=magnitude_error,flag=check_flags,mag=mag)
des_cols = (
    [f'CLASS_STAR_{band}' for band in bandList] + 
    [f'FLAGS_{band}' for band in bandList] + 
    ['RA','DEC','COADD_OBJECT_ID'] + 
    [f'SPREAD_MODEL_{band}' for band in bandList] + 
    [f'WAVG_MAG_PSF_{band}' for band in bandList] + 
    [f'WAVG_MAGERR_PSF_{band}' for band in bandList]
)
max_obj_deviation = 0.2
pm_speed_min = 1000 #units are milliarcseconds per year
pm_speed_max = 10**5
milliarc_degree_conversion = 1/(1000*3600)
print("Defined local vars.")

Defined local vars.


In [16]:
importlib.reload(hpms)

print("Done")

Done


In [5]:
config_dict = {

    # Catalog Specific Parameters:
    "catalog": lsdb.read_hats(DES_DIR, margin_cache=DES_MARGIN_CACHE_DIR),
    "id_col_name": "COADD_OBJECT_ID",
    "mag_cols": [f'WAVG_MAG_PSF_{band}' for band in ['I', 'G']],
    "mag_err_cols": [f'WAVG_MAGERR_PSF_{band}' for band in ['I', 'G']],

    # Filtering Specific Parameters:
    "query_string": query_string,
    "xmatch_max_neighbors": 100,
    "max_neighbor_dist": 18,
    "min_neighbors": 3,
    "k": 2,
    "max_obj_deviation": 0.2,

    # Additional Pipeline Parameters:
    "debug_mode": True
}
config = hpms.PipelineConfig_from_dict(config_dict)

print("Defined Config")

Defined Config


In [6]:
print(config.min_neighbors)

3


In [7]:
%%time
pm_filter_xmatch = xmatch_catalog.query(f'{pm_speed_max**2} >(pmra_gaia**2 + pmdec_gaia**2) > {pm_speed_min**2}')
with Client():
    df = pm_filter_xmatch.compute()

df

2025-07-17 16:30:02,544 - distributed.worker - ERROR - Failed to communicate with scheduler during heartbeat.
Traceback (most recent call last):
  File "/ocean/projects/phy210048p/jpassos/conda-venvs/lsdb-main/lib/python3.12/site-packages/distributed/comm/tcp.py", line 226, in read
    frames_nosplit_nbytes_bin = await stream.read_bytes(fmt_size)
                                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
tornado.iostream.StreamClosedError: Stream is closed

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/ocean/projects/phy210048p/jpassos/conda-venvs/lsdb-main/lib/python3.12/site-packages/distributed/worker.py", line 1269, in heartbeat
    response = await retry_operation(
               ^^^^^^^^^^^^^^^^^^^^^^
  File "/ocean/projects/phy210048p/jpassos/conda-venvs/lsdb-main/lib/python3.12/site-packages/distributed/utils_comm.py", line 416, in retry_operation
    return await retry(
           ^^^^^^^^^^^^
  File "/

CPU times: user 31.3 s, sys: 840 ms, total: 32.1 s
Wall time: 1min 20s


,CLASS_STAR_G_des,CLASS_STAR_R_des,CLASS_STAR_I_des,CLASS_STAR_Z_des,CLASS_STAR_Y_des,FLAGS_G_des,FLAGS_R_des,FLAGS_I_des,FLAGS_Z_des,FLAGS_Y_des,RA_des,DEC_des,COADD_OBJECT_ID_des,SPREAD_MODEL_G_des,SPREAD_MODEL_R_des,SPREAD_MODEL_I_des,SPREAD_MODEL_Z_des,SPREAD_MODEL_Y_des,WAVG_MAG_PSF_G_des,WAVG_MAG_PSF_R_des,WAVG_MAG_PSF_I_des,WAVG_MAG_PSF_Z_des,WAVG_MAG_PSF_Y_des,WAVG_MAGERR_PSF_G_des,WAVG_MAGERR_PSF_R_des,WAVG_MAGERR_PSF_I_des,WAVG_MAGERR_PSF_Z_des,WAVG_MAGERR_PSF_Y_des,NEPOCHS_G_des,NEPOCHS_R_des,NEPOCHS_I_des,NEPOCHS_Z_des,NEPOCHS_Y_des,source_id_gaia,ra_gaia,dec_gaia,pmra_gaia,pmdec_gaia,phot_g_mean_mag_gaia,phot_bp_mean_mag_gaia,phot_rp_mean_mag_gaia,_dist_arcsec
_healpix_29,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
613844718320588,0.880538,0.963898,0.983112,0.979363,0.595527,0,0,0,0,0,43.194227,1.924013,1343006866,-0.002108,-0.000071,0.00106,-0.000283,0.000484,22.757801,21.253117,20.287706,19.843111,19.703493,0.026349,0.008517,0.007532,0.006415,0.019259,7,7,4,7,6,1227712107314688,43.195868,1.928423,1400.291765,-515.645438,13.919317,14.903977,12.939081,16.939085
613854791118803,0.513397,0.702804,0.690331,0.556331,0.001543,1,1,1,1,1,43.198956,1.924676,1343006899,-0.004575,0.009738,0.01514,-0.017007,0.021923,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,0,0,0,0,0,1227712107314688,43.195868,1.928423,1400.291765,-515.645438,13.919317,14.903977,12.939081,17.476786
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3289742749783385503,0.999789,0.999768,0.999853,0.999168,0.22071,3,3,3,3,3,322.705738,-40.715997,926149359,0.002088,0.007195,0.01138,0.014347,0.004569,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,0,0,0,0,0,6579485487647973888,322.704773,-40.714388,1046.642165,-1396.306284,11.778023,13.375048,10.548469,6.364448
3289742749975291194,0.030484,0.097534,0.844041,0.842068,0.628413,3,3,3,3,3,322.70503,-40.714652,926147617,0.021968,0.046018,0.04153,0.04223,0.048809,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,0,0,0,0,0,6579485487647973888,322.704773,-40.714388,1046.642165,-1396.306284,11.778023,13.375048,10.548469,1.182722


In [8]:
df_no_dupes = df[~df['source_id_gaia'].duplicated(keep='first')]

gaia_ids = df['source_id_gaia']

#dropping because otherwise produces error when performing .apply below
df_no_dupes = df_no_dupes.drop('source_id_gaia', axis=1)

df_no_dupes

,CLASS_STAR_G_des,CLASS_STAR_R_des,CLASS_STAR_I_des,CLASS_STAR_Z_des,CLASS_STAR_Y_des,FLAGS_G_des,FLAGS_R_des,FLAGS_I_des,FLAGS_Z_des,FLAGS_Y_des,RA_des,DEC_des,COADD_OBJECT_ID_des,SPREAD_MODEL_G_des,SPREAD_MODEL_R_des,SPREAD_MODEL_I_des,SPREAD_MODEL_Z_des,SPREAD_MODEL_Y_des,WAVG_MAG_PSF_G_des,WAVG_MAG_PSF_R_des,WAVG_MAG_PSF_I_des,WAVG_MAG_PSF_Z_des,WAVG_MAG_PSF_Y_des,WAVG_MAGERR_PSF_G_des,WAVG_MAGERR_PSF_R_des,WAVG_MAGERR_PSF_I_des,WAVG_MAGERR_PSF_Z_des,WAVG_MAGERR_PSF_Y_des,NEPOCHS_G_des,NEPOCHS_R_des,NEPOCHS_I_des,NEPOCHS_Z_des,NEPOCHS_Y_des,ra_gaia,dec_gaia,pmra_gaia,pmdec_gaia,phot_g_mean_mag_gaia,phot_bp_mean_mag_gaia,phot_rp_mean_mag_gaia,_dist_arcsec
_healpix_29,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
613844718320588,0.880538,0.963898,0.983112,0.979363,0.595527,0,0,0,0,0,43.194227,1.924013,1343006866,-0.002108,-0.000071,0.00106,-0.000283,0.000484,22.757801,21.253117,20.287706,19.843111,19.703493,0.026349,0.008517,0.007532,0.006415,0.019259,7,7,4,7,6,43.195868,1.928423,1400.291765,-515.645438,13.919317,14.903977,12.939081,16.939085
1153482605725265461,0.844888,0.845371,0.844807,0.845333,0.844827,3,3,3,3,3,1.386363,-37.369781,1043295027,-0.015229,0.040008,0.037639,0.034238,0.050002,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,0,0,0,0,0,1.383284,-37.367744,5633.438088,-2334.721273,7.682494,8.802319,6.61627,11.461677
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3263270778643414534,0.530344,0.565081,0.25516,0.771278,0.506456,1,1,1,1,1,352.558439,-47.618479,1010242107,-0.001722,-0.005203,0.00807,0.031992,-0.013712,-99.0,24.201725,-99.0,-99.0,-99.0,-99.0,0.241926,-99.0,-99.0,-99.0,0,1,0,0,0,352.563657,-47.61685,-562.651573,-973.694222,15.216914,18.328148,13.734485,13.953139
3289742749783385503,0.999789,0.999768,0.999853,0.999168,0.22071,3,3,3,3,3,322.705738,-40.715997,926149359,0.002088,0.007195,0.01138,0.014347,0.004569,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,0,0,0,0,0,322.704773,-40.714388,1046.642165,-1396.306284,11.778023,13.375048,10.548469,6.364448


In [9]:
def algo_found_pm(row, des_cols, cone_search_rad):    
    # Filter DES around PM of interest:
    catalog = (
        lsdb.read_hats(DES_DIR, columns=des_cols, margin_cache=DES_MARGIN_CACHE_DIR)
        .cone_search(ra=row['ra_gaia'], dec=row['dec_gaia'], radius_arcsec=cone_search_rad)
    )
    config.catalog = catalog
    
    with Client():
        filtered_df = hpms.execute_pipeline(config).compute()
    if not filtered_df.empty:
        min_idx = filtered_df['kth_min_deviation'].idxmin()
        best_row = filtered_df.loc[min_idx]
        return pd.Series({
            'kth_min_deviation': best_row['kth_min_deviation'],
            'max_obj_distance': best_row['max_obj_distance'],
            'max_mag_diff': best_row['max_mag_diff'],
            'aligned_neighbors': best_row['aligned_neighbors']
        })
    else:
        print('EMPTY')
        return pd.Series({
            'kth_min_deviation': np.nan,
            'max_obj_distance': np.nan,
            'max_mag_diff': np.nan,
            'aligned_neighbors': np.nan
        })

In [17]:
res = df_no_dupes.apply(func=algo_found_pm, axis=1, des_cols=des_cols, cone_search_rad=25)
res

Length after neighbors filter: 104
Length after kth star filter: 0
EMPTY
Length after neighbors filter: 0
Length after kth star filter: 0
EMPTY
Length after neighbors filter: 12
Length after kth star filter: 0
EMPTY
Length after neighbors filter: 12
Length after kth star filter: 0
EMPTY
Length after neighbors filter: 105
Length after kth star filter: 0
EMPTY
Length after neighbors filter: 0
Length after kth star filter: 0
EMPTY
Length after neighbors filter: 0
Length after kth star filter: 0
EMPTY
Length after neighbors filter: 0
Length after kth star filter: 0
EMPTY
Length after neighbors filter: 213
Length after kth star filter: 0
EMPTY
Length after neighbors filter: 0
Length after kth star filter: 0
EMPTY
Length after neighbors filter: 0
Length after kth star filter: 0
EMPTY
Length after neighbors filter: 0
Length after kth star filter: 0
EMPTY
Length after neighbors filter: 0
Length after kth star filter: 0
EMPTY
Length after neighbors filter: 29
Length after kth star filter: 0
EMP

,kth_min_deviation,max_obj_distance,max_mag_diff,aligned_neighbors
_healpix_29,,,,
613844718320588,NaN,NaN,NaN,NaN
1153482605725265461,NaN,NaN,NaN,NaN
...,...,...,...,...
3263270778643414534,NaN,NaN,NaN,NaN
3289742749783385503,NaN,NaN,NaN,NaN


In [18]:
res.dropna()

,kth_min_deviation,max_obj_distance,max_mag_diff,aligned_neighbors
_healpix_29,,,,
2451171327961423729,0.125761,14.150352,4.596519,"[RA: 13.067564, DEC: -62.031401, mags: [-99.0, -99.0], mag_errs: [-99.0, -99.0], deviation: inf, RA: 13.073349, DEC: -62.031192, mags: [-99.0, 18.925487518310547], mag_errs: [-99.0, 0.0020345954690128565], deviation: inf, RA: 13.07592, DEC: -62.031097, mags: [-99.0, 18.948963165283203], mag_errs: [-99.0, 0.0036583596374839544], deviation: inf, RA: 13.074734, DEC: -62.031177, mags: [-99.0, 18.902997970581055], mag_errs: [-99.0, 0.003902954049408436], deviation: inf]"
2503116016090087218,0.009369,6.712114,3.336334,"[RA: 11.341313, DEC: -33.497952, mags: [18.123525619506836, 19.659683227539062], mag_errs: [0.0029473849572241306, 0.006811248138546944], deviation: inf, RA: 11.340758, DEC: -33.497576, mags: [18.118959426879883, 19.66592788696289], mag_errs: [0.002000729087740183, 0.004199234303086996], deviation: inf, RA: 11.342493, DEC: -33.498752, mags: [18.116546630859375, 19.632564544677734], mag_errs: [0.003134039230644703, 0.007033675909042358], deviation: inf, RA: 11.341839, DEC: -33.498305, mags: [18.115943908691406, 19.652009963989258], mag_errs: [0.0016613374464213848, 0.003399771172553301], deviation: inf]"
2592746699573430477,0.016478,8.841134,3.045273,"[RA: 42.81831, DEC: -3.889707, mags: [-99.0, -99.0], mag_errs: [-99.0, -99.0], deviation: inf, RA: 42.818014, DEC: -3.889203, mags: [17.116657257080078, 22.706378936767578], mag_errs: [0.0011425362899899483, 0.05949027091264725], deviation: inf, RA: 42.817064, DEC: -3.887589, mags: [-99.0, -99.0], mag_errs: [-99.0, -99.0], deviation: inf, RA: 42.817701, DEC: -3.888661, mags: [17.086204528808594, 22.60576629638672], mag_errs: [0.0011771972058340907, 0.04488782957196236], deviation: inf]"
3206298005501909246,0.046316,19.238638,10.512352,"[RA: 331.077916, DEC: -56.79445, mags: [17.970745086669922, 24.229557037353516], mag_errs: [0.0028097128961235285, 0.2106461524963379], deviation: inf, RA: 331.080109, DEC: -56.795217, mags: [17.991500854492188, -99.0], mag_errs: [0.002536272630095482, -99.0], deviation: inf, RA: 331.071881, DEC: -56.792344, mags: [17.886377334594727, 23.579452514648438], mag_errs: [0.0012036056723445654, 0.11593939363956451], deviation: inf, RA: 331.075861, DEC: -56.793716, mags: [-99.0, -99.0], mag_errs: [-99.0, -99.0], deviation: inf]"
